In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import spearmanr
import seaborn as sns
import matplotlib.pyplot as plt
import gc

adata_full = sc.read_h5ad('/home/vlad/vaeae/data_bio/covid_portal_with_raw.h5ad', backed='r')

mono_labels = ['CD14_mono', 'CD83_CD14_mono', 'CD16_mono', 'C1_CD16_mono', 'Mono_prolif']
mono_mask = adata_full.obs['full_clustering'].isin(mono_labels)

adata_mono = adata_full[mono_mask].to_memory()
del adata_full
gc.collect()

In [2]:
print(f"monocytes: {adata_mono.n_obs}")
print("features:", adata_mono.var['feature_types'].value_counts())

adata_rna = adata_mono[:, adata_mono.var['feature_types'] == 'Gene Expression'].copy()
adata_prot = adata_mono[:, adata_mono.var['feature_types'] == 'Antibody Capture'].copy()

genes_to_hide = ['S100A8', 'S100A9', 'S100A12', 'MS4A7', 'CD14', 'FCGR3A', 'C1QA', 'C1QB', 'HLA-DRA', 'CSF1R']
mask = ~adata_rna.var_names.isin(genes_to_hide)
adata_rna = adata_rna[:, mask].copy()

if "counts" in adata_rna.layers:
    adata_rna.layers["counts"] = adata_rna.layers["counts"].copy()
elif "raw" in adata_rna.layers:
    adata_rna.layers["counts"] = adata_rna.layers["raw"].copy()
else:
    adata_rna.layers["counts"] = adata_rna.X.copy()

monocytes: 142258
features: feature_types
Gene Expression     24737
Antibody Capture      192
Name: count, dtype: int64


In [3]:
adata_rna.write_h5ad('/home/vlad/vaeae/data_bio/covid_mono_rna_prepared.h5ad', compression='gzip')
adata_prot.write_h5ad('/home/vlad/vaeae/data_bio/covid_mono_prot.h5ad', compression='gzip')

import gc
del adata_mono
del adata_rna
del adata_prot
gc.collect()

print("Сохранено!")

Сохранено!


=====================================================================

In [1]:
import scanpy as sc

adata_rna = sc.read_h5ad('/home/vlad/vaeae/data_bio/covid_mono_rna_prepared.h5ad')
adata_prot = sc.read_h5ad('/home/vlad/vaeae/data_bio/covid_mono_prot.h5ad')

print(adata_rna)

AnnData object with n_obs × n_vars = 142258 × 24727
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'counts', 'raw'


In [2]:
sc.pp.normalize_total(adata_rna, target_sum=1e4)
sc.pp.log1p(adata_rna)

sc.pp.highly_variable_genes(adata_rna, n_top_genes=2000, flavor="seurat_v3", layer="counts")
sc.tl.pca(adata_rna, n_comps=30, use_highly_variable=True)
sc.external.pp.harmony_integrate(adata_rna, 'sample_id')

/home/vlad/miniconda3/envs/bioml/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
2026-10-28 20:03:33,204 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-10-28 20:03:44,508 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-28 20:03:45,606 - harmonypy - INFO - Iteration 1 of 10
2026-10-28 20:05:13,437 - harmonypy - INFO - Iteration 2 of 10
2026-10-28 20:06:37,133 - harmonypy - INFO - Iteration 3 of 10
2026-10-28 20:07:58,800 - harmonypy - INFO - Iteration 4 of 10
2026-10-28 20:09:20,595 - harmonypy - INFO - Iteration 5 of 10
2026-10-28 20:10:41,798 - harmonypy - INFO - Iteration 6 of 10
2026-10-28 20:11:55,171 - harmonypy - INFO - Iteration 7 

In [3]:
import pandas as pd

harmony_df = pd.DataFrame(
    adata_rna.obsm['X_pca_harmony'], 
    index=adata_rna.obs_names
)
harmony_df.to_csv('/home/vlad/vaeae/data_bio/harmony_coords_mono.csv', index=True, index_label='cell_id')

labels_df = pd.DataFrame({
    'full_clustering': adata_rna.obs['full_clustering'].values
}, index=adata_rna.obs_names)
labels_df.to_csv('/home/vlad/vaeae/data_bio/cluster_labels_mono.csv', index=True, index_label='cell_id')

print("CSV сохранены.")

CSV сохранены.


In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import gc

path = '/home/vlad/vaeae/data_bio/covid_portal_with_raw.h5ad'
mono_labels = ['CD14_mono', 'CD83_CD14_mono', 'CD16_mono', 'C1_CD16_mono', 'Mono_prolif']

adata_full = sc.read_h5ad(path, backed='r')

mono_mask = adata_full.obs['full_clustering'].isin(mono_labels).values

umap_key = 'X_umap' if 'X_umap' in adata_full.obsm else 'umap'
umap_all = np.asarray(adata_full.obsm[umap_key])
obs_all = adata_full.obs.copy()

umap_mono = umap_all[mono_mask, :2]

meta_cols = [c for c in ['full_clustering', 'sample', 'donor', 'cell_type']
             if c in obs_all.columns]

umap_df = pd.DataFrame(
    umap_mono,
    index=obs_all.index[mono_mask],
    columns=['UMAP1', 'UMAP2']
).join(obs_all.loc[mono_mask, meta_cols])

out = '/home/vlad/vaeae/data_bio/covid_mono_umap.csv'
umap_df.to_csv(out)
print(f"Сохранено {len(umap_df)} строк -> {out}")

del adata_full
gc.collect()

Сохранено 142258 строк -> /home/vlad/vaeae/data_bio/covid_mono_umap.csv


103

In [ ]:
import numpy as np
import scanpy as sc
adata_full = sc.read_h5ad('/home/vlad/vaeae/data_bio/covid_portal_with_raw.h5ad', backed='r')

mono_labels = ['CD14_mono', 'CD83_CD14_mono', 'CD16_mono', 'C1_CD16_mono', 'Mono_prolif']
mono_mask = adata_full.obs['full_clustering'].isin(mono_labels).values

H_orig = np.asarray(adata_full.obsm['X_pca_harmony'])[mono_mask]
ids = adata_full.obs_names[mono_mask]

print(H_orig.shape)

(142258, 50)


NameError: name 'ref' is not defined

In [4]:
import pandas as pd

ref = pd.read_csv('/home/vlad/vaeae/data_bio/cluster_labels_mono.csv')
assert (ids.values == ref['cell_id'].values).all()

pd.DataFrame(H_orig, index=ids).to_csv(
    '/home/vlad/vaeae/data_bio/harmony_coords_mono_authors.csv',
    index=True, index_label='cell_id')

adata_full.file.close()